# 实验5：寄存器堆设计


## 一、实验说明

### 1.1 实验背景

寄存器堆（Register File）是CPU中用于暂存运算数据和中间结果的高速存储单元，是CPU最核心的部件之一。在MIPS架构中，寄存器堆包含32个32位通用寄存器（编号$0~$31），其中$0寄存器硬连线为0，写入无效，读取始终返回0。寄存器堆需要同时支持读和写两种操作，且通常是多端口的——可以同时读取两个寄存器的值（用于ALU的两个操作数），同时写入一个寄存器（存储结果）。

本实验基于Logisim仿真设计环境，设计简化的MIPS寄存器文件，进行读写功能验证。Logisim作为一款开源逻辑电路仿真工具，以其直观的图形界面和丰富的组件库，成为国内外多所高校计算机硬件实验的首选平台。


### 1.2 实验环境准备

1. **软件环境**：
   - Logisim：[参见预装软件指南](https://gitee.com/totalcontrol/hustzc/blob/master/README.md)
   - 操作系统：Windows / macOS / Linux（需安装Java运行环境）

2. **硬件环境**：
   - 安装有Logisim软件的计算机一台

3. **实验文件准备**：
   - 下载[预装软件](https://gitee.com/totalcontrol/hustzc/tree/master/%E9%A2%84%E8%A3%85%E8%BD%AF%E4%BB%B6)
   - 准备Logisim项目文件[storage.circ](https://gitee.com/totalcontrol/hustzc/tree/master/5.%E5%AD%98%E5%82%A8%E7%B3%BB%E7%BB%9F%E5%AE%9E%E9%AA%8C)


## 二、实验任务

### 2.1 任务描述

使用Logisim平台构建一个简化的MIPS寄存器文件，具体要求如下：

1. **读端口**：包含两个读寄存器号输入端口`RD1`和`RD2`，分别指定待读出数据的两个寄存器（可能相同，也可能不同），读出的数据分别通过`data1`和`data2`两个输出端口获得。

2. **写端口**：包含一个写寄存器号输入端口`WR`，指定待写入数据的寄存器，数据通过`dataIn`端口送入寄存器。

3. **写入控制**：数据写入时需要写使能信号有效（`regWrite=1`），并在时钟`clk`上升沿到来时写入数据。

4. **位宽规格**：数据线位宽为32bit，寄存器号端口位宽为5bit（完整MIPS规格），时钟和写使能信号位宽为1bit。**实验时实现4~8个寄存器的读写控制即可**。

### 2.2 学习目标

1. 理解MIPS寄存器文件的基本概念与工作原理
2. 掌握Logisim中寄存器、多路选择器、译码器等组件的使用方法
3. 具备寄存器堆架构设计能力，理解组合逻辑与时序逻辑电路的协同工作
4. 能够独立完成寄存器堆的搭建、功能验证与调试

## 三、任务准备

### 3.1 前置知识

**（1）MIPS寄存器堆的基本架构**

MIPS寄存器文件是CPU的“短期记忆”，容量极小但访问速度极快。典型MIPS寄存器堆包含：
- 32个通用寄存器（$0~$31），每个32位宽
- 2个读端口 + 1个写端口
- 特殊寄存器：$0恒为零

**（2）寄存器堆的时序特性**

寄存器堆的读出过程使用**组合逻辑**完成——当地址发生改变，新数据在若干传播延迟后就在RD上出现，不需要时钟参与。而**写入过程仅在时钟上升沿发生**，因此寄存器堆整体是同步时序电路。

**（3）核心组件的功能与区别**

| 组件 | 功能 | 在寄存器堆中的作用 |
|------|------|-------------------|
| 寄存器（Register） | 存储单元，带使能端控制是否响应时钟 | 每个寄存器存储一个32位数据 |
| 多路选择器（MUX） | 从多路输入中选择一路输出 | **读操作**：根据寄存器编号选择对应的寄存器输出 |
| 译码器（Decoder） | 将输入编码转换为独热码输出 | **写操作**：根据寄存器编号生成对应寄存器的使能信号 |

> **关键理解**：读用MUX（从多个源中选一个数据出来），写用Decoder（把一个数据送到多个目标中的一个）。


## 四、任务实施

### 4.1 总体设计思路

寄存器堆的核心设计围绕三个问题展开：
1. **写哪里？** —— 由`WR`端口通过译码器选择目标寄存器
2. **读哪里？** —— 由`RD1`和`RD2`端口通过多路选择器选择源寄存器
3. **何时写？** —— 由`regWrite`使能信号和时钟上升沿共同控制

### 4.2 方案选择

两种主流设计方案对比：

| 方案 | 优点 | 缺点 | 适用场景 |
|------|------|------|---------|
| 多路选择器方案 | 布线简单、直观易懂 | 扩展性差（寄存器增多时MUX规模庞大） | 4~8个寄存器的小规模实验 |
| 译码器+三态门方案 | 扩展性好、适合大规模 | 布线复杂、需理解三态门 | 32个寄存器的完整实现 |

**建议**：初学者先用**多路选择器方案**，理解透彻后再尝试译码器+三态门方案。

### 4.3 实施步骤
具体实验步骤：[第2关：MIPS寄存器文件设计](https://www.educoder.net/tasks/853zahj4/3769162/2em7tizpvklf?coursesId=853zahj4)

## 五、实验总结

本实验通过Logisim平台完成了简化MIPS寄存器堆的设计与实现，主要收获如下：

**1. 理论与实践的结合**：将MIPS架构中寄存器堆的抽象概念转化为可运行的电路模块，直观理解了寄存器堆“双端口读、单端口写”的工作机制。

**2. 组合逻辑与时序逻辑的协同**：深刻体会到寄存器堆中读操作（组合逻辑）与写操作（时序逻辑）的不同特性——读操作不需要时钟参与，而写操作严格依赖时钟上升沿和使能信号。

**3. 组件选型与方案权衡**：通过多路选择器方案和译码器+三态门方案的对比，理解了不同设计方案在布线复杂度、扩展性等方面的优劣。

**4. 调试能力的锻炼**：在搭建和验证过程中，通过观察Logisim的线路颜色、设置寄存器初始值、设计测试用例等方法，逐步定位并解决了连线错误、使能端遗漏、$0寄存器处理等典型问题。

**5. 为后续CPU设计打下基础**：寄存器堆是CPU数据通路的核心部件，本实验的成功完成为后续单周期MIPS CPU的设计与实现奠定了坚实基础。
